# 03 - Inferencia de posible tratamiento

Ejecuta el modelo local de posibles tratamientos/hallazgos cuando el peso entrenado este disponible.
La ruta esperada esta centralizada en `src/model_registry.py`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)

In [ ]:
from model_registry import get_model_spec

spec = get_model_spec("treatment")
print(spec.display_name)
print("Modelo:", spec.weights)
print("Disponible:", spec.exists)
print("Clases esperadas:", len(spec.expected_classes))

In [ ]:
IMAGE_PATH = PROJECT_ROOT / "data" / "samples" / "sample-19.jpg"
OUTPUT_DIR = PROJECT_ROOT / "results" / "notebooks" / "treatment_inference"

if not spec.exists:
    print("Modelo pendiente: copia el peso entrenado en", spec.weights)
else:
    cmd = [
        sys.executable,
        "src/predict_dental_xray.py",
        "--model-role", "treatment",
        "--image", str(IMAGE_PATH),
        "--output-dir", str(OUTPUT_DIR),
        "--device", "cpu",
        "--imgsz", "640",
        "--conf", "0.25",
    ]
    subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
from IPython.display import Image, display

output_image = OUTPUT_DIR / "treatment_detector" / f"{IMAGE_PATH.stem}_treatment_detector_labeled.jpg"
output_report = OUTPUT_DIR / "treatment_detector" / f"{IMAGE_PATH.stem}_treatment_detector_report.csv"

print("Imagen:", output_image)
print("Reporte:", output_report)
if output_image.exists():
    display(Image(filename=str(output_image)))
if output_report.exists():
    print(output_report.read_text(encoding="utf-8").splitlines()[:8])